In [1]:
# Calculate and save MDA8

In [2]:
import os
import xarray as xr
import numpy as np
import json
from utils.utils import get_scenario_config
from utils.utils import standardise_latlon

In [3]:
def load_file_list(DIR, filename):
    file_path = os.path.join(DIR, filename)
    with open(file_path, "r") as f:
        data = json.load(f)
    return data["files"]

In [4]:
# Conversion functions
def kgkg_to_ppb(data):
    # kg/kg -> ppb (multiply by 6.0345e8)
    return data * 6.0345e8


def molmol_to_ppb(data):
    # mol/mol -> ppb (multiply by 1e9)
    return data * 1e9


# Map model -> variable name + conversion
MODEL_CONFIG = {
    "UKESM1": {
        "var": "mass_fraction_of_ozone_in_air",
        "convert": kgkg_to_ppb},
    "CESM2": {
        "var": "O3_SRF",
        "convert": molmol_to_ppb},
}


def load_ozone_ppb(ds, model_name):
    config = MODEL_CONFIG[model_name]
    var_name = config["var"]
    converter = config["convert"]

    data = ds[var_name]
    converted = converter(data)
    converted.attrs["units"] = "ppb"
    return converted

In [5]:
# === Return frequency string e.g '3-hourly' for xarray with time coord ===
def time_frequency(data):
    step = data.time.diff("time").median()
    # convert to hours
    step_hours = step / np.timedelta64(1, "h")
    # handle whole days if nicer
    if step_hours % 24 == 0:
        return f"{int(step_hours/24)}-daily"
    else:
        return f"{int(step_hours)}-hourly"

In [6]:
# === Processing function ===
def calculate_monthly_mean_8hrdailymax(start_date, end_date, o3_surf, calendar):
    daterange = xr.date_range(start_date, end_date, calendar=calendar, use_cftime=True)

    MDA8 = xr.DataArray(  # Maximum Daily 8hr Average O3
        np.nan,
        dims=["time", "lat", "lon"],
        coords={"time": daterange, "lat": o3_surf.lat, "lon": o3_surf.lon},
    )

    for i in range(len(daterange)):
        date = daterange[i].strftime("%Y-%m-%d")
        # print(f"Processing {date}")
        o3_day = o3_surf.sel(time=slice(date + " 00:00:00", date + " 23:00:00"))

        # if 3-hourly snapshot, interpolate to hourly
        if time_frequency(o3_day) == "3-hourly":
            hourly_day = xr.date_range(
                date + " 00:00:00",
                date + " 23:00:00",
                freq="h", calendar=calendar, use_cftime=True)
            o3_day = o3_day.interp(time=hourly_day)

        o3_rolling = o3_day.rolling(time=8).mean()
        MDA8[i, :, :] = o3_rolling.max("time")

    monthly_mean = MDA8.resample(time="ME").mean()
    return monthly_mean

In [7]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "UKESM1"
scenario = "hist"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/ozone/file_paths/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/ozone/MDA8/"


# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    file_list = load_file_list(FILE_DIR, f"file_list_{scenario}_{ens_num}.json")
    monthly_means = []

    for f in file_list:
        if not os.path.exists(f):
            raise ValueError(f"Missing: {f}")

        print(f"Reading {os.path.basename(f)}")
        ds = xr.open_dataset(f)
        da = standardise_latlon(load_ozone_ppb(ds, model))

        start_date = str(da.time[0].values)[:10]  # e.g. yyyy-mm-dd
        end_date = str(da.time[-1].values)[:10]
        end_time = str(da.time[-1].values)[11:16]  # e.g. hh:mm
        midnight = "00:00"

        if end_time == midnight:
            print("changing final time step")
            # making final time step 23:00
            end_date = str(ds.time[-2].values)[:10]

        model_calendar = da.time.encoding.get("calendar")

        mm = calculate_monthly_mean_8hrdailymax(
            start_date,
            end_date,
            da,
            model_calendar)

        # Trim to start_year - end_year
        mm = mm.sel(time=slice(str(years.start), str(years.stop)))

        monthly_means.append(mm)

    if monthly_means:
        combined = xr.concat(monthly_means, dim="time")

        dates = f"{years.start}0101-{years.stop}1231"

        out_file = f"MDA8_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        description = ("MDA8: 8hr Daily Maximum Calculates the 8-hour "
                       "daily maximum surface ozone concentration - scripts "
                       "by A.F. Wells (2025)")
        combined.attrs["description"] = description
        combined.attrs["units"] = "ppb"
        combined.attrs["ensemble_number"] = ens_num
        combined.attrs["scenario"] = scenario
        combined.attrs["model"] = model
        combined.to_netcdf(out_path)

print("All processing complete.")


Processing hist, Ensemble 01
Reading sfo3_AERhr_UKESM1-0-LL_historical_r1i1p1f2_gn_199001010030-199912302330.nc


KeyError: "No variable named 'mass_fraction_of_ozone_in_air'. Variables on the dataset include ['time', 'time_bnds', 'lat', 'lat_bnds', 'lon', 'lon_bnds', 'sfo3']"